# 🔍 Data Validation Example

Validate data and handle errors using `quarantine` mode.

## Scenario
- **Unique Email**: Duplicate emails are excluded
- **Valid Age**: Age must be between 0 and 120
- **User Type**: Must not be null

> **Note**: Quarantine mode excludes invalid rows from output. It does not save quarantined rows to a separate folder.

In [ ]:
!pip install -q "mlprep-rust==0.3.1" pandas pyarrow

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path.cwd() / 'outputs'
BASE.mkdir(exist_ok=True)

np.random.seed(42)
n_rows = 100
df = pd.DataFrame({
    'id': range(1, n_rows + 1),
    'email': [f'user{i}@example.com' for i in range(n_rows)],
    'age': np.random.randint(15, 60, size=n_rows),
    'user_type': np.random.choice(['admin', 'user', 'guest'], size=n_rows)
})

# Introduce bad data
df.loc[0, 'email'] = 'duplicate@example.com'
df.loc[1, 'email'] = 'duplicate@example.com'
df.loc[2, 'age'] = -5
df.loc[3, 'age'] = 150
df.loc[4, 'user_type'] = np.nan

df.to_csv('dirty_data.csv', index=False)
print('Generated dirty_data.csv with intentional errors')
print('\n🚨 Bad rows:')
print(df.iloc[:5][['id', 'email', 'age', 'user_type']])

In [ ]:
pipeline_yaml = '''name: data_validation
inputs:
  - path: dirty_data.csv
    format: csv

steps:
  - type: validate
    checks:
      columns:
        - name: email
          unique: true
        - name: age
          range: [0, 120]
        - name: user_type
          not_null: true
    mode: quarantine

outputs:
  - path: outputs/clean_output.parquet
    format: parquet
'''

with open('pipeline.yaml', 'w') as f:
    f.write(pipeline_yaml)
print(pipeline_yaml)

In [ ]:
!mlprep run pipeline.yaml --streaming --memory-limit 1GB

In [ ]:
import os

if os.path.exists('outputs/clean_output.parquet'):
    clean_df = pd.read_parquet('outputs/clean_output.parquet')
    print(f'✅ Clean output: {len(clean_df)} rows')
    print(clean_df.head(10))
else:
    print('❌ clean_output.parquet not found')

In [ ]:
input_df = pd.read_csv('dirty_data.csv')

print('📊 Data Quality Summary')
print('='*50)
print(f'Input rows: {len(input_df)}')
if os.path.exists('outputs/clean_output.parquet'):
    clean_df = pd.read_parquet('outputs/clean_output.parquet')
    print(f'Clean output rows: {len(clean_df)}')
    print(f'Rows excluded: {len(input_df) - len(clean_df)}')
    print('\n✅ Validation Results:')
    print(f'  All emails unique: {clean_df["email"].is_unique}')
    print(f'  All ages in [0, 120]: {((clean_df["age"] >= 0) & (clean_df["age"] <= 120)).all()}')
    print(f'  No null user_type: {clean_df["user_type"].notna().all()}')